# ACER v3.1 — Adaptation & Evaluation Analysis

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
ROOT=Path.cwd()
if ROOT.name=='notebooks': ROOT=ROOT.parent
sys.path.append(str(ROOT))
from app.engine import load_requirements, load_system, assess
from app.adaptation import load_tactics
from app.agents import ComplianceOrchestrator
reqs=load_requirements(ROOT/'data/requirements/demo_requirements.yaml')
tactics=load_tactics(ROOT/'data/tactics/tactics.yaml')
system=load_system(ROOT/'data/systems/recruitment_noncompliant.yaml')


In [ ]:
before=assess(system,reqs)
final_system,history=ComplianceOrchestrator(reqs,tactics).run(system)
after=assess(final_system,reqs)
print('Before:',before.overall_status)
print('After:',after.overall_status)
pd.DataFrame(history)


In [ ]:
from app.scenarios import write_scenarios
generated=ROOT/'data/systems/generated'
write_scenarios(generated,n=30,seed=42)
rows=[]
for p in sorted(generated.glob('scenario-*.yaml')):
    s=load_system(p)
    b=assess(s,reqs)
    f,h=ComplianceOrchestrator(reqs,tactics).run(s)
    a=assess(f,reqs)
    rows.append({
        'system_id':s.id,
        'before_violations':sum(r.status=='FAIL' for r in b.results),
        'after_violations':sum(r.status=='FAIL' for r in a.results),
        'steps':sum(x.get('accepted',False) for x in h),
        'success':a.overall_status=='COMPLIANT'
    })
scenario_df=pd.DataFrame(rows)
scenario_df


In [ ]:
print('Adaptation success rate:',round(100*scenario_df.success.mean(),2),'%')
print('Mean accepted steps:',round(scenario_df.steps.mean(),2))


In [ ]:
ax=scenario_df[['before_violations','after_violations']].mean().plot(kind='bar',figsize=(7,4))
ax.set_title('Mean violations before vs after adaptation')
ax.set_ylabel('Average failed requirements')
plt.tight_layout()
plt.show()


Next research step: replace synthetic requirements with source-traceable, independently validated regulatory requirements; then evaluate LLM/MBSE/agentic variants against ground truth.